# ContractGuard AI — Executed Capstone Evidence

**Program:** SDAIA Academy — Advanced Agentic AI Systems Engineering  
**Cohort/session:** June 2026  
**Project:** Secure, observable, resumable multi-agent vendor-contract audit platform

This notebook executes and preserves evidence for all six rubric deliverables: real,
schema-validated tool use and named reasoning patterns, framework-managed graph
orchestration, role-specialized agents, security and observability, durable
checkpoint/HITL/cloud artifacts, and professional documentation.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Project root:', PROJECT_ROOT)
print('Python:', sys.version.split()[0])


## 1. Execute the complete security, retry, HITL, restart, and output-validation demonstration

The runner contains hard assertions. Any missing rubric behavior causes the cell to fail.
It executes a real prompt-injection attack, a safe contract, a high-risk contract with a
simulated tool timeout, a Reflexion re-search loop, a durable human interrupt, a fresh
service restart, human approval, an output-schema revision loop, PII masking, and artifact
storage.


In [ ]:
result = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts' / 'run_capstone_demo.py')],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
    check=True,
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)


## 2. Rubric assertion summary

In [ ]:
summary = json.loads((PROJECT_ROOT / 'evidence' / 'run_summary.json').read_text())
proof = pd.DataFrame(
    [{'requirement': key.replace('_', ' '), 'passed': value} for key, value in summary['proof'].items()]
)
display(proof)
assert proof['passed'].all()
print('All proof assertions:', bool(proof['passed'].all()))


## 3. Deliverable 1 — Agentic reasoning and real function tools

The Coordinator implements **Plan-and-Execute**. Every function is described by a
Pydantic-generated JSON Schema and invoked through the MCP-style registry. The default
reproducible mode uses a deterministic schema-aware router; the same interface supports
provider-native function calling with Gemini, OpenRouter, or Groq. Each call records a
concise **ReAct** triple: rationale/Thought, Action, and Observation. The Quality Reviewer
implements **Reflexion/self-critique**, and the coordinator-to-specialist topology is
**Hierarchical Delegation**.


In [ ]:
low = json.loads((PROJECT_ROOT / 'evidence' / '02_low_risk_completed.json').read_text())
tool_rows = []
observations = {o['call_id']: o for o in low['tool_observations']}
for call in low['tool_calls']:
    obs = observations[call['call_id']]
    tool_rows.append({
        'agent': call['agent'],
        'tool': call['tool_name'],
        'decision_source': call['decision_source'],
        'protocol': call['protocol'],
        'provider': call.get('model_provider') or '',
        'model': call.get('model_name') or '',
        'live_llm': call.get('used_live_llm', False),
        'rationale': call['rationale'],
        'status': obs['status'],
        'latency_ms': round(obs['latency_ms'], 3),
        'observation': obs['summary'],
    })
display(pd.DataFrame(tool_rows))
print('Named reasoning patterns:', summary['reasoning_patterns'])
print('Shared state keys carried across steps:', len(low.keys()))
print('Reasoner modes:', low.get('reasoner_modes'))
assert len(low['tool_calls']) >= 6
assert 'ReAct' in {d.get('pattern') for d in low['decision_trace']}
assert any(c['decision_source'] in {'offline_schema_router', 'llm_function_call'} for c in low['tool_calls'])
assert any(c['protocol'] in {'mcp_json_schema', 'provider_native_function_call'} for c in low['tool_calls'])
policy_calls = [c for c in low['tool_calls'] if c['tool_name'] == 'search_policy_knowledge_base']
assert policy_calls
assert all(c['decision_source'] == 'offline_schema_router' for c in policy_calls)
assert all(c['protocol'] == 'mcp_json_schema' for c in policy_calls)


## 4. Deliverable 2 — Genuine graph orchestration with conditions and loops

The graph is built by the real `transitions.Machine` finite-state orchestration framework
(`transitions==0.9.3`). Conditions decide branches; shared state is read and updated by
node callbacks; three bounded cycles support tool retry, Reflexion/re-search, and report
revision. The generated specification explicitly proves that this is not a linear chain.


In [ ]:
graph = json.loads((PROJECT_ROOT / 'evidence' / 'graph_spec.json').read_text())
print('Framework:', graph['framework'])
print('Package/version:', graph['framework_package'], graph['framework_version'])
print('Nodes/edges/conditional:', graph['node_count'], graph['edge_count'], graph['conditional_edge_count'])
print('Branching nodes:', graph['branching_nodes'])
print('Loops:')
for loop in graph['loops']:
    print(' -', loop)
edge_frame = pd.DataFrame(graph['edges'])[['trigger', 'source', 'dest', 'conditions', 'before']].fillna('')
display(edge_frame)
assert graph['node_count'] >= 10
assert graph['is_linear_chain'] is False
assert graph['has_cycles'] is True
assert graph['has_conditional_routing'] is True
assert any(edge['source'] == edge['dest'] for edge in graph['edges'])
assert any(edge.get('conditions') for edge in graph['edges'])


## 5. Failure paths — actual tool retry and Reflexion re-search

The first policy search deliberately raises a simulated timeout. The failed
`ToolObservation` is retained, the conditional self-loop fires, and the next attempt
succeeds. A separate quality critique routes back to research.


In [ ]:
paused = json.loads((PROJECT_ROOT / 'evidence' / '03_high_risk_paused_for_human.json').read_text())
failed_tools = [o for o in paused['tool_observations'] if o['status'] == 'error']
print('Failed tool observations:', json.dumps(failed_tools, indent=2))
print('Policy retry count:', paused['policy_retry_count'])
print('Quality re-plan count:', paused['quality_retry_count'])
print('Research node visits:', paused['node_history'].count('researching'))
print('Node path:', ' -> '.join(paused['node_history']))
assert failed_tools
assert paused['policy_retry_count'] >= 1
assert paused['quality_retry_count'] >= 1
assert paused['node_history'].count('researching') >= 3


## 6. Deliverable 3 — Multi-agent role specialization and structured communication

These are separate agent objects/classes, not personas concatenated into one prompt.
Messages contain sender, recipient, message type, content, payload, and timestamp.


In [ ]:
messages = pd.DataFrame(paused['agent_messages'])
agent_summary = messages.groupby('sender').agg(
    messages=('sender', 'size'),
    recipients=('recipient', lambda values: ', '.join(sorted(set(values))))
).reset_index()
display(agent_summary)
display(messages[['sender', 'recipient', 'message_type', 'content']].tail(12))
print('Coordination strategy:', summary['coordination_strategy'])
assert messages['sender'].nunique() >= 7
assert {'sender', 'recipient', 'message_type', 'payload'}.issubset(messages.columns)


## 7. Deliverable 4 — Security guardrails and structured observability

The malicious uploaded contract is inspected before the tool registry is reachable.
Every agent has an explicit tool allowlist, tool arguments use strict Pydantic schemas,
contract paths are confined to configured roots, and thread IDs have a safe format.
Before optional cloud-model use, raw contract excerpts are excluded and PII is masked.
The output guardrail also masks PII and validates a strict Pydantic schema. Monitoring is
JSONL + Prometheus, not print statements.


In [ ]:
blocked = json.loads((PROJECT_ROOT / 'evidence' / '01_prompt_injection_blocked.json').read_text())
print('Attack terminal status:', blocked['status'])
print('Detected reason:', blocked['blocked_reason'])
print('Tool calls after block:', len(blocked['tool_calls']))
print('Blocked path:', ' -> '.join(blocked['node_history']))
print('Per-agent tool permissions:')
print(json.dumps(summary['tool_interface']['agent_tool_permissions'], indent=2))
assert blocked['status'] == 'blocked'
assert len(blocked['tool_calls']) == 0
assert summary['proof']['per_agent_tool_permissions_configured'] is True


In [ ]:
log_path = PROJECT_ROOT / 'evidence' / 'execution_log.jsonl'
logs = [json.loads(line) for line in log_path.read_text().splitlines() if line.strip()]
log_frame = pd.DataFrame(logs)
print('Structured log events:', len(log_frame))
print('Event types:', sorted(log_frame['event'].dropna().unique()))
display(log_frame[['timestamp', 'event', 'thread_id', 'node', 'tool', 'latency_ms']].tail(15).fillna(''))

metrics_text = (PROJECT_ROOT / 'evidence' / 'metrics_before_restart.prom').read_text()
metric_names = sorted({line.split('{', 1)[0].split(' ', 1)[0] for line in metrics_text.splitlines() if line and not line.startswith('#')})
print('Prometheus metric series (sample):', metric_names[:20])
assert 'tool_call_failed' in set(log_frame['event'])
assert 'guardrail_blocked' in set(log_frame['event'])
assert 'human_interrupt' in set(log_frame['event'])
assert 'contractguard_tool_calls_total' in metrics_text
assert 'contractguard_llm_calls_total' in metrics_text


## 8. Deliverable 5 — Persistent checkpoint, real HITL pause/resume, and cloud artifact

A high-risk contract stops at `awaiting_approval`. The first service object is closed.
A fresh service object opens the same SQLite database, reloads the thread, applies a human
decision, and continues from the paused node.


In [ ]:
loaded = json.loads((PROJECT_ROOT / 'evidence' / '04_checkpoint_loaded_after_restart.json').read_text())
final = json.loads((PROJECT_ROOT / 'evidence' / '05_high_risk_resumed_and_completed.json').read_text())
print('Node loaded after restart:', loaded['node'])
print('Interrupt payload:', json.dumps(loaded['state']['interrupt_payload'], indent=2))
print('Final status:', final['status'])
print('Human decision:', final['approval_status'], '-', final['approver'])
print('Output revision count:', final['report_revision_count'])
print('PII redactions:', final['pii_redactions'])
print('Artifact URI:', final['artifact_uri'])
assert loaded['node'] == 'awaiting_approval'
assert final['status'] == 'completed'
assert final['approval_status'] == 'approved'
assert final['report_revision_count'] >= 1
assert final['pii_redactions'] >= 3


In [ ]:
cloud_files = [
    'Dockerfile', 'docker-compose.yml', 'deploy/prometheus.yml', 'src/contractguard/api.py',
    'scripts/docker_minio_smoke.py', '.github/workflows/ci.yml'
]
cloud_evidence = []
for relative in cloud_files:
    path = PROJECT_ROOT / relative
    cloud_evidence.append({'artifact': relative, 'exists': path.exists(), 'bytes': path.stat().st_size if path.exists() else 0})
display(pd.DataFrame(cloud_evidence))
assert all(item['exists'] for item in cloud_evidence)


## 9. Final masked compliance report

In [ ]:
report_path = Path(final['artifact_uri'].removeprefix('file://'))
report_text = report_path.read_text()
print(report_text[:6000])
assert '[REDACTED_EMAIL]' in report_text
assert '[REDACTED_PHONE]' in report_text
assert '[REDACTED_NATIONAL_ID]' in report_text
assert 'nora.alqahtani@example.com' not in report_text


## 10. Automated tests

The test suite covers direct/indirect injection, PII masking, real tool search, strict tool
schemas, per-agent tool permissions, safe thread IDs, contract-path isolation, offline
schema routing, mocked provider-native function calls, provider fallback, pre-model data
minimization/redaction, low-risk completion, graph retry, restart persistence, HITL resume,
output revision, artifact storage, API-key enforcement, and FastAPI endpoints.


In [ ]:
tests = subprocess.run(
    [sys.executable, '-m', 'pytest', '--override-ini', 'addopts=', '-q', '--color=no'],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
    check=True,
    env={**os.environ, 'PYTHONPATH': str(PROJECT_ROOT / 'src')},
)
print(tests.stdout)
(PROJECT_ROOT / 'evidence' / 'pytest_results.txt').write_text(tests.stdout + tests.stderr)
collected = subprocess.run(
    [sys.executable, '-m', 'pytest', '--collect-only', '-q', '--color=no'],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
    check=True,
    env={**os.environ, 'PYTHONPATH': str(PROJECT_ROOT / 'src')},
)
test_count = sum(
    int(line.rsplit(':', 1)[1].strip())
    for line in collected.stdout.splitlines()
    if line.startswith('tests/') and line.rsplit(':', 1)[1].strip().isdigit()
)
print('Collected tests:', test_count)
assert test_count >= 16


## 11. Documentation and submission completeness

- Professional README with setup, API keys, expected outputs, deployment, evidence index,
  training attribution, and SDAIA Academy link.
- Technical architecture using nodes, edges, state, agents, tools, conditions, loops, and
  checkpointers, plus a GitHub-renderable Mermaid graph.
- Rubric traceability, security model, API reference, tests, `.gitignore`, Docker/Compose,
  MinIO runtime smoke test, pre-publication gate, CI workflow, and third-party notices.
- Executed notebook and captured JSON/log/metric/report evidence.

The only external submission step is pushing this prepared Git repository to the
trainee's chosen GitHub repository.


In [ ]:
required = [
    'README.md', 'README.md', '.gitignore', '.env.example',
    'docs/architecture.md', 'docs/agent_graph.md', 'docs/rubric_traceability.md',
    'docs/security.md', 'docs/api.md', 'Dockerfile', 'docker-compose.yml',
    '.github/workflows/ci.yml', 'scripts/prepublish_check.py',
    'scripts/run_live_function_call_demo.py', 'scripts/docker_minio_smoke.py',
    'THIRD_PARTY_NOTICES.md',
]
rows = [{'file': item, 'exists': (PROJECT_ROOT / item).exists()} for item in required]
display(pd.DataFrame(rows))
assert all(row['exists'] for row in rows)
print('Executed capstone notebook completed successfully.')
